# Authorized recovery of the interrupted quadratic-Bezier comparison

This notebook completes the one preserved interrupted comparison. It pins the validated recovery implementation, extracts both authorizations from their exact commits, re-runs the complete tests, performs a read-only stable-state preflight, byte-verifies and reuses 17 completed units, quarantines the one partial unit without overwrite, executes exactly 19 missing units, rebuilds one aggregate after all 36 units verify, and applies the blinded-review gate.

Use a fresh CPU runtime and run code cells 1–6 in order. Cell 5 is the recovery execution. **Do not interrupt Cell 5.** If your internet disconnects or Colab says reconnecting, leave the runtime alone and reconnect later; do not press Stop. If any cell fails, preserve everything, disconnect nothing until the error is recorded, and do not rerun.

In [ ]:
from pathlib import Path
import json
import shutil
import subprocess

REPO_DIR = Path('/content/latent-stroke-dynamics')
REPO_URL = 'https://github.com/Navid111/latent-stroke-dynamics.git'
BRANCH = 'quadratic-bezier-extension'
FROZEN_RUNNER_COMMIT = '398a2bfb7bd65ed8b4bbc93fb8cc05564f7f3c1b'
RECOVERY_IMPLEMENTATION_COMMIT = '46e0c6396f0425ed84812e8fbeef9ed675ef53e9'
ORIGINAL_AUTHORIZATION_COMMIT = 'cc857407ed431c5583fd9e1c02a0ba619a8c187a'
RECOVERY_VALIDATION_EVIDENCE_COMMIT = '5cc2e6c98bb58b6ad917b593b97dbd359033fe75'
RECOVERY_AUTHORIZATION_COMMIT = '76b6d53bddaaa60880e7c7f1eaffd1392c9ece25'
ORIGINAL_AUTHORIZATION_REPO_PATH = 'configs/quadratic-bezier-execution-authorization-2026-09-04.json'
RECOVERY_AUTHORIZATION_REPO_PATH = 'configs/quadratic-bezier-recovery-authorization-2026-09-05.json'
ORIGINAL_AUTHORIZATION_PATH = Path('/content/quadratic-bezier-execution-authorization-2026-09-04.json')
RECOVERY_AUTHORIZATION_PATH = Path('/content/quadratic-bezier-recovery-authorization-2026-09-05.json')

if Path('/content/drive/MyDrive').exists():
    raise RuntimeError('STOP: use a fresh runtime before mounting Drive in Cell 4.')
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
clone = subprocess.run(
    ['git', 'clone', '--quiet', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)],
    capture_output=True,
    text=True,
)
if clone.returncode != 0:
    print(clone.stdout)
    print(clone.stderr)
    raise RuntimeError('Public repository clone failed.')
original_authorization_text = subprocess.check_output(
    ['git', 'show', f'{ORIGINAL_AUTHORIZATION_COMMIT}:{ORIGINAL_AUTHORIZATION_REPO_PATH}'],
    cwd=REPO_DIR,
    text=True,
)
recovery_authorization_text = subprocess.check_output(
    ['git', 'show', f'{RECOVERY_AUTHORIZATION_COMMIT}:{RECOVERY_AUTHORIZATION_REPO_PATH}'],
    cwd=REPO_DIR,
    text=True,
)
ORIGINAL_AUTHORIZATION_PATH.write_text(original_authorization_text, encoding='utf-8')
RECOVERY_AUTHORIZATION_PATH.write_text(recovery_authorization_text, encoding='utf-8')
subprocess.run(
    ['git', 'checkout', '--quiet', '--detach', RECOVERY_IMPLEMENTATION_COMMIT],
    cwd=REPO_DIR,
    check=True,
)
observed_commit = subprocess.check_output(
    ['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True
).strip()
assert observed_commit == RECOVERY_IMPLEMENTATION_COMMIT
assert json.loads(original_authorization_text)['authorized_runner_commit'] == FROZEN_RUNNER_COMMIT
assert json.loads(recovery_authorization_text)['recovery_implementation_commit'] == RECOVERY_IMPLEMENTATION_COMMIT
assert subprocess.check_output(
    ['git', 'status', '--short'], cwd=REPO_DIR, text=True
).strip() == ''
print('CELL 1 COMPLETE — EXACT RUNNER, RECOVERY, AND BOTH AUTHORIZATIONS PINNED')
print('frozen runner:', FROZEN_RUNNER_COMMIT)
print('recovery implementation:', RECOVERY_IMPLEMENTATION_COMMIT)
print('recovery authorization commit:', RECOVERY_AUTHORIZATION_COMMIT)

In [ ]:
import re
import subprocess
import time
from pathlib import Path

subprocess.run(
    ['python', '-m', 'pip', 'install', '-q', '-e', '.'],
    cwd=REPO_DIR,
    check=True,
)
test_started = time.monotonic()
test_run = subprocess.run(
    ['python', '-m', 'pytest', '-q'],
    cwd=REPO_DIR,
    capture_output=True,
    text=True,
)
test_duration_seconds = time.monotonic() - test_started
EXECUTION_PYTEST_LOG = Path('/content/quadratic_bezier_recovery_execution_pytest.txt')
EXECUTION_PYTEST_LOG.write_text(test_run.stdout + test_run.stderr, encoding='utf-8')
print(test_run.stdout)
if test_run.stderr:
    print(test_run.stderr)
assert test_run.returncode == 0, 'The complete test suite failed; do not recover.'
match = re.search(r'(\d+) passed', test_run.stdout + test_run.stderr)
assert match is not None and int(match.group(1)) == 217
print('CELL 2 COMPLETE — COMPLETE 217-TEST SUITE PASSED')

In [ ]:
import os
import sys

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR / 'src'))
from latent_stroke_dynamics.quadratic_bezier_comparison import (
    validate_execution_authorization,
    validate_target_freeze,
)
from latent_stroke_dynamics.quadratic_bezier_recovery import (
    inspect_stable_interrupted_state,
    load_recovery_plan,
    validate_only_recovery_report,
    validate_recovery_authorization,
)

PLAN_PATH = REPO_DIR / 'configs/quadratic-bezier-interrupted-recovery-plan-2026-09-05.json'
FREEZE_PATH = REPO_DIR / 'configs/quadratic-bezier-target-freeze-2026-09-04.json'
plan = load_recovery_plan(PLAN_PATH)
freeze = validate_target_freeze(FREEZE_PATH)
original_authorization = validate_execution_authorization(
    ORIGINAL_AUTHORIZATION_PATH,
    source_commit=FROZEN_RUNNER_COMMIT,
    freeze_path=FREEZE_PATH,
)
recovery_authorization = validate_recovery_authorization(
    RECOVERY_AUTHORIZATION_PATH,
    recovery_implementation_commit=RECOVERY_IMPLEMENTATION_COMMIT,
    plan_path=PLAN_PATH,
)
validation = validate_only_recovery_report(
    repo_root=REPO_DIR,
    plan_path=PLAN_PATH,
    freeze_path=FREEZE_PATH,
    expected_head=RECOVERY_IMPLEMENTATION_COMMIT,
)
assert validation['status'] == 'quadratic_bezier_recovery_implementation_valid_no_outputs_unauthorized'
assert recovery_authorization['status'] == 'authorized_for_single_interrupted_comparison_recovery'
assert recovery_authorization['recovery_execution_authorized'] is True
assert recovery_authorization['fresh_comparison_execution_allowed'] is False
assert recovery_authorization['expected_missing_run_count'] == 19
print('CELL 3 COMPLETE — FROZEN INPUTS AND ONE-TIME RECOVERY AUTHORIZATION VALIDATED')

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
OUTPUT_PARENT = Path('/content/drive/MyDrive/latent-stroke-dynamics-rgb')
OUTPUT_DIR = OUTPUT_PARENT / 'quadratic-bezier-fixed-comparison-v1'
INCOMPLETE_DIR = OUTPUT_PARENT / 'quadratic-bezier-fixed-comparison-v1.incomplete'
preflight = inspect_stable_interrupted_state(
    output_dir=OUTPUT_DIR,
    plan=plan,
    freeze=freeze,
    original_authorization=original_authorization,
)
assert preflight['completed_run_count'] == 17
assert preflight['partial_run_count'] == 1
assert preflight['not_started_run_count'] == 18
assert preflight['missing_run_count'] == 19
print('CELL 4 COMPLETE — STABLE DRIVE STATE RE-VERIFIED READ ONLY')
print('byte-verified completed units:', preflight['completed_run_count'])
print('preserved partial units:', preflight['partial_run_count'])
print('authorized missing units:', preflight['missing_run_count'])

In [ ]:
import json
from pathlib import Path
import subprocess
import time

EXECUTION_LOG = Path('/content/quadratic_bezier_recovery_execution_log.txt')
command = [
    'python',
    'run_quadratic_bezier_recovery.py',
    '--execute-recovery',
    '--repo-root', str(REPO_DIR),
    '--plan', str(PLAN_PATH),
    '--freeze', str(FREEZE_PATH),
    '--output-dir', str(OUTPUT_DIR),
    '--original-authorization', str(ORIGINAL_AUTHORIZATION_PATH),
    '--recovery-authorization', str(RECOVERY_AUTHORIZATION_PATH),
    '--recovery-implementation-commit', RECOVERY_IMPLEMENTATION_COMMIT,
]
started = time.monotonic()
print('STARTING AUTHORIZED MISSING-ONLY RECOVERY — DO NOT INTERRUPT')
print('If the browser disconnects, do not press Stop; reconnect and let this continue.')
with EXECUTION_LOG.open('w', encoding='utf-8') as log_handle:
    process = subprocess.Popen(
        command,
        cwd=REPO_DIR,
        stdout=log_handle,
        stderr=subprocess.STDOUT,
        text=True,
    )
    try:
        while process.poll() is None:
            elapsed = time.monotonic() - started
            print(f'Recovery still running — elapsed {elapsed / 60:.1f} minutes')
            time.sleep(30)
    except KeyboardInterrupt:
        process.terminate()
        process.wait()
        print('Recovery was interrupted. Preserve all Drive state and do not rerun.')
        raise
return_code = process.returncode
if return_code != 0:
    tail = EXECUTION_LOG.read_text(encoding='utf-8')[-5000:]
    print(tail)
    raise RuntimeError('Recovery failed. Preserve all Drive state and stop; do not rerun.')
execution_handoff = json.loads(EXECUTION_LOG.read_text(encoding='utf-8'))
assert execution_handoff['status'] == 'quadratic_bezier_recovery_completed_blind_gate_applied'
assert execution_handoff['completed_run_count'] == 36
assert execution_handoff['reused_completed_run_count'] == 17
assert execution_handoff['executed_during_recovery_run_count'] == 19
assert OUTPUT_DIR.is_dir()
assert not INCOMPLETE_DIR.exists()
print('CELL 5 COMPLETE — ONE INTERRUPTED COMPARISON RECOVERED AND FINALIZED')
print(f'recovery elapsed minutes: {(time.monotonic() - started) / 60:.2f}')

In [ ]:
from hashlib import sha256
import json
from pathlib import Path
from google.colab import files

SUMMARY_PATH = OUTPUT_DIR / 'aggregate_summary.json'
SUMMARY_SHA_PATH = OUTPUT_DIR / 'aggregate_summary.sha256'
summary = json.loads(SUMMARY_PATH.read_text(encoding='utf-8'))
observed_summary_sha = sha256(SUMMARY_PATH.read_bytes()).hexdigest()
assert SUMMARY_SHA_PATH.read_text(encoding='utf-8').strip() == observed_summary_sha
assert summary['status'] == 'quadratic_bezier_fixed_comparison_complete'
assert summary['completion_mode'] == 'verified_interrupted_attempt_recovery'
assert summary['source_commit'] == FROZEN_RUNNER_COMMIT
assert summary['recovery_implementation_commit'] == RECOVERY_IMPLEMENTATION_COMMIT
assert summary['completed_run_count'] == 36
assert summary['completed_pair_count'] == 18
assert summary['reused_completed_run_count'] == 17
assert summary['executed_during_recovery_run_count'] == 19
assert summary['integrity_passed'] is True
assert summary['training_performed'] is False
assert summary['learned_model_used'] is False
assert summary['closed_experiments_changed'] is False
verified_artifacts = 0
for relative_path, expected_sha in summary['artifact_sha256'].items():
    artifact = OUTPUT_DIR / relative_path
    assert artifact.is_file(), relative_path
    assert sha256(artifact.read_bytes()).hexdigest() == expected_sha, relative_path
    verified_artifacts += 1
qualitative_required = bool(summary['quantitative_decision']['qualitative_review_required'])
blind_handoff = {
    'status': 'quadratic_bezier_recovery_blind_handoff_ready',
    'frozen_runner_commit': FROZEN_RUNNER_COMMIT,
    'recovery_implementation_commit': RECOVERY_IMPLEMENTATION_COMMIT,
    'recovery_validation_evidence_commit': RECOVERY_VALIDATION_EVIDENCE_COMMIT,
    'recovery_authorization_commit': RECOVERY_AUTHORIZATION_COMMIT,
    'aggregate_summary_sha256': observed_summary_sha,
    'target_set_sha256': summary['target_set_sha256'],
    'completed_run_count': summary['completed_run_count'],
    'completed_pair_count': summary['completed_pair_count'],
    'reused_completed_run_count': summary['reused_completed_run_count'],
    'executed_during_recovery_run_count': summary['executed_during_recovery_run_count'],
    'integrity_passed': summary['integrity_passed'],
    'verified_artifact_count': verified_artifacts,
    'qualitative_review_required': qualitative_required,
    'quantitative_values_withheld_for_blinding': qualitative_required,
}
BLIND_HANDOFF_PATH = Path('/content/quadratic_bezier_recovery_blind_handoff.json')
BLIND_HANDOFF_PATH.write_text(
    json.dumps(blind_handoff, indent=2, sort_keys=True) + '\n',
    encoding='utf-8',
)
files.download(str(BLIND_HANDOFF_PATH))
if qualitative_required:
    files.download(str(OUTPUT_DIR / 'blinded_review_montage.png'))
    files.download(str(OUTPUT_DIR / 'blinded_review_sheet.csv'))
    print('CELL 6 COMPLETE — BLINDED REVIEW IS REQUIRED')
    print('Return only the blind handoff JSON, blinded montage, and blank review sheet.')
    print('Do not open the mapping, aggregate summary, metrics, plots, or execution log.')
else:
    for name in [
        'aggregate_summary.json',
        'aggregate_summary.sha256',
        'quantitative_decision.json',
        'target_seed_pair_metrics.csv',
        'mean_512_mse_by_primitive.png',
        'per_target_curve_ratio.png',
        'aggregate_progress_by_primitive.png',
        'recovery_audit/initial_state_manifest.json',
        'recovery_audit/recovery_journal.json',
    ]:
        files.download(str(OUTPUT_DIR / name))
    files.download(str(EXECUTION_PYTEST_LOG))
    files.download(str(EXECUTION_LOG))
    print('CELL 6 COMPLETE — NO BLINDED REVIEW GATE REQUIRED; NUMERICAL HANDOFF DOWNLOADED')
print('verified artifacts:', verified_artifacts)

## Cell 7 — reveal only after blinded review

If Cell 6 says blinded review is required, stop and return the three blind-review files first. After the review is completed and recorded, set the flag below to `True` and run only Cell 7. If no blinded review is required, Cell 6 already downloaded the numerical handoff and this cell is unnecessary.

In [ ]:
from google.colab import files

REVEAL_AFTER_BLINDED_REVIEW = False
if qualitative_required:
    if not REVEAL_AFTER_BLINDED_REVIEW:
        raise RuntimeError('STOP: complete and record the blinded review before revealing identities or metrics.')
    reveal_names = [
        'aggregate_summary.json',
        'aggregate_summary.sha256',
        'quantitative_decision.json',
        'target_seed_pair_metrics.csv',
        'mean_512_mse_by_primitive.png',
        'per_target_curve_ratio.png',
        'aggregate_progress_by_primitive.png',
        'blinded_mapping_do_not_open_before_review.json',
        'recovery_audit/initial_state_manifest.json',
        'recovery_audit/recovery_journal.json',
    ]
    for name in reveal_names:
        files.download(str(OUTPUT_DIR / name))
    files.download(str(EXECUTION_PYTEST_LOG))
    files.download(str(EXECUTION_LOG))
    print('CELL 7 COMPLETE — POST-REVIEW NUMERICAL AND IDENTITY HANDOFF DOWNLOADED')
else:
    print('Cell 7 not needed: Cell 6 already downloaded the numerical handoff.')